# Vectorstores and Embeddings

In [4]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-groq sentence-transformers transformers chromadb panel param

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94

In [1]:
from google.colab import files
uploaded = files.upload()  # click "Choose file", pick document_splits.pkl from C:\Users\User\Documents\Law chatbot\law-chatbot-langchain\data\processed\

Saving document_splits.json to document_splits.json


In [2]:
import json
from langchain_core.documents import Document

with open("document_splits.json", encoding="utf-8") as f:
    records = json.load(f)

splits = [Document(page_content=r["page_content"], metadata=r["metadata"]) for r in records]
print(f"Loaded {len(splits)} chunks")

Loaded 25286 chunks


## Embeddings

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

#embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [10]:
sentence1 = "المستاجر ملزم بدفع الاجرة في موعدها"
sentence2 = "يجب على المستاجر سداد بدل الايجار في الوقت المحدد"
sentence3 = "الطقس اليوم ممطر وبارد"

embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)

In [11]:
import numpy as np

print("sentence1 vs sentence2 (both about rent payment):", np.dot(embedding1, embedding2))
print("sentence1 vs sentence3 (unrelated):              ", np.dot(embedding1, embedding3))

sentence1 vs sentence2 (both about rent payment): 0.9138109409207551
sentence1 vs sentence3 (unrelated):               0.3666112359962347


## Vectorstore — Chroma, persisted locally

In [9]:
persist_directory = "/content/chroma"

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 8},  # caps how many texts get embedded together at once
)

BATCH = 50
vectordb = None
for i in range(0, len(splits), BATCH):
    batch = splits[i:i + BATCH]
    if vectordb is None:
        vectordb = Chroma.from_documents(documents=batch, embedding=embedding, persist_directory=persist_directory)
    else:
        vectordb.add_documents(batch)
    print(f"{min(i + BATCH, len(splits))}/{len(splits)} embedded", flush=True)

print("DONE. Collection count:", vectordb._collection.count())


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

50/25286 embedded
100/25286 embedded
150/25286 embedded
200/25286 embedded
250/25286 embedded
300/25286 embedded
350/25286 embedded
400/25286 embedded
450/25286 embedded
500/25286 embedded
550/25286 embedded
600/25286 embedded
650/25286 embedded
700/25286 embedded
750/25286 embedded
800/25286 embedded
850/25286 embedded
900/25286 embedded
950/25286 embedded
1000/25286 embedded
1050/25286 embedded
1100/25286 embedded
1150/25286 embedded
1200/25286 embedded
1250/25286 embedded
1300/25286 embedded
1350/25286 embedded
1400/25286 embedded
1450/25286 embedded
1500/25286 embedded
1550/25286 embedded
1600/25286 embedded
1650/25286 embedded
1700/25286 embedded
1750/25286 embedded
1800/25286 embedded
1850/25286 embedded
1900/25286 embedded
1950/25286 embedded
2000/25286 embedded
2050/25286 embedded
2100/25286 embedded
2150/25286 embedded
2200/25286 embedded
2250/25286 embedded
2300/25286 embedded
2350/25286 embedded
2400/25286 embedded
2450/25286 embedded
2500/25286 embedded
2550/25286 embedded


In [14]:
print("Total chunks in vectorstore:", vectordb._collection.count())
print("Expected:", len(splits))

Total chunks in vectorstore: 25286
Expected: 25286


In [8]:
#from langchain_chroma import Chroma

#persist_directory = "../data/chroma/"

#vectordb = Chroma.from_documents(
 #   documents=splits,
  #  embedding=embedding,
   # persist_directory=persist_directory,
#)
#print(vectordb._collection.count())

OutOfMemoryError: CUDA out of memory. Tried to allocate 4.30 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.99 GiB is free. Including non-PyTorch memory, this process has 10.57 GiB memory in use. Of the allocated memory 10.43 GiB is allocated by PyTorch, and 16.48 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

### Similarity search sanity check

In [12]:
question = "هل يجوز فصل عامل تغيب عن العمل بدون انذار؟"
docs = vectordb.similarity_search(question, k=3)
len(docs)

3

In [13]:
docs[0].page_content[:500]

'جلسة 3 من ابريل سنة 2017 برئاسة المستشار علي يوسف منصور وعضوية المستشارين/ يحيي فتحي شافعي يمامة ،عبدالله يعقوب عبدالرحمن ، محمد حسن البوعينين ، عبدالمنعم ابراهيم الشهاوي ( 122 ) الطعن رقم 211 لسنة 2015 (1- 2) حكم. عمل. (1) لصاحب العمل فصل العامل دون اخطار او تعويض . شرطه . تغيبه عن العمل بدون سبب مشروع اكثر من عشرين يوما منفصلة خلال السنة الواحدة او اكثر من عشرة ايام متوالية وان يسبق الفصل انذار كتابي من صاحب العمل بعد غيابه عشرة ايام في الحالة الاولي وانقطاعه خمسة ايام في الحالة الثانية . م 13'

In [15]:
question = "هل يجوز فصل عامل تغيب عن العمل بدون انذار؟"  # "can an employer dismiss a worker who was absent without notice?"
docs = vectordb.similarity_search(question, k=3)

for d in docs:
    print(d.metadata)
    print(d.page_content[:300])
    print("---")

{'source': 'sjc', 'case_type': 'مدني', 'doc_id': '211 M 2015 K 122'}
جلسة 3 من ابريل سنة 2017 برئاسة المستشار علي يوسف منصور وعضوية المستشارين/ يحيي فتحي شافعي يمامة ،عبدالله يعقوب عبدالرحمن ، محمد حسن البوعينين ، عبدالمنعم ابراهيم الشهاوي ( 122 ) الطعن رقم 211 لسنة 2015 (1- 2) حكم. عمل. (1) لصاحب العمل فصل العامل دون اخطار او تعويض . شرطه . تغيبه عن العمل بدون سبب م
---
{'doc_id': '291 M 2014 K 85', 'source': 'sjc', 'case_type': 'مدني'}
جلسة 14 من يونيو سنة 2016 برئاسة المستشار د. طه عبد المولي طه وعضوية المستشارين/ نادر السيد علي عبدالمطلب ، ابراهيم محمد المرصفاوي ، عبدالله يعقوب عبدالرحمن (85) الطعن رقم 291 لسنة 2014 (1-3) تعويض. حكم" عيوب التدليل: القصور في التسبيب". دفاع. عقد. عمل. محكمة الموضوع . (1) لصاحب العمل انهاء عقد العم
---
{'source': 'sjc', 'case_type': 'مدني', 'doc_id': '747 M 2025 K 0'}
جلسة 10 من فبراير سنة 20 26 برئاسة السيد القاضي احمد علي يحي الوكيل بالمحكمه، والساده القضاه خالد احمد المدفع وحسام الدين شاكر الوكيلين بالمحكمة ود عاصم رمضان يونس ومنير محمد امين. ( ) ا

In [17]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree('/content/chroma', '/content/drive/MyDrive/law_chatbot_chroma', dirs_exist_ok=True)
print("Vectorstore backed up to Drive")

Mounted at /content/drive
Vectorstore backed up to Drive
